In [2]:
from pathlib import Path
import shutil

src_dir = Path(r"E:\with_adapt")
dst_dir = Path(r"C:\修論研究\Iizumi\wheat")
dst_dir.mkdir(parents=True, exist_ok=True)

copied = 0
for f in src_dir.rglob("*"):
    if f.is_file() and f.suffix.lower() == ".nc" and "wheat" in f.name.lower():
        shutil.copy2(f, dst_dir / f.name)
        copied += 1

print(f"コピー完了: {copied} 件")


コピー完了: 150 件


In [3]:
import sys
print(sys.executable)  # どのPythonを使っているか確認
!{sys.executable} -m pip install xarray netCDF4 h5netcdf


c:\Users\tsuda\AppData\Local\Programs\Python\Python313\python.exe
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.4 MB 308.8 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.4 MB 308.8 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.4 MB 308.8 kB/s eta 0:00:03
   ---------------------- ----------------- 0.8/1.4 MB 327.7 kB/s eta 0:00:02
   ---------------------- ----------------- 0.8/1.4 MB 327.7 kB/s eta 0:00:02
   ---------------------- ----------------- 0.8/1.4 MB 327.7 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install -U xarray h5netcdf h5py netCDF4


   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   --- ---------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import tempfile
import shutil
import uuid
from netCDF4 import Dataset

src = Path(r"C:\修論研究\Iizumi\wheat\cygma_gfdl-esm4_ssp126_ssp1_wheat_irri_yld_1981_2100.nc").resolve()
print("src exists:", src.exists())
print("src:", src)

tmp_nc = Path(tempfile.gettempdir()) / f"{src.stem}_{uuid.uuid4().hex}.nc"
print("tmp:", tmp_nc)

# copy2 ではなく copyfile を使う（メタ情報処理を省いてロック衝突を減らす）
shutil.copyfile(src, tmp_nc)

try:
    with Dataset(str(tmp_nc), mode="r") as nc:
        print("\n=== Data model ===")
        print(nc.data_model)

        print("\n=== Dimensions ===")
        for dname, dim in nc.dimensions.items():
            print(f"{dname}: size={len(dim)}, unlimited={dim.isunlimited()}")

        print("\n=== Variables ===")
        for vname, var in nc.variables.items():
            print(f"{vname}: dims={var.dimensions}, shape={var.shape}, dtype={var.dtype}")

        print("\n=== Global attributes ===")
        for aname in nc.ncattrs():
            print(f"{aname}: {getattr(nc, aname)}")
finally:
    try:
        tmp_nc.unlink(missing_ok=True)
    except Exception:
        pass


src exists: True
src: C:\修論研究\Iizumi\wheat\cygma_gfdl-esm4_ssp126_ssp1_wheat_irri_yld_1981_2100.nc
tmp: C:\Users\tsuda\AppData\Local\Temp\cygma_gfdl-esm4_ssp126_ssp1_wheat_irri_yld_1981_2100_2cace70b2747479da70cf374a3598f56.nc

=== Data model ===
NETCDF4

=== Dimensions ===
lon: size=720, unlimited=False
lat: size=360, unlimited=False
time: size=120, unlimited=False

=== Variables ===
lon: dims=('lon',), shape=(720,), dtype=float64
lat: dims=('lat',), shape=(360,), dtype=float64
time: dims=('time',), shape=(120,), dtype=float64
var: dims=('time', 'lat', 'lon'), shape=(120, 360, 720), dtype=float32

=== Global attributes ===


# 排出シナリオー社会経済シナリオをそろえたうえで、それぞれについて気候モデルに関してensembleをとる